# M7 - Flow re-extraction from the original CSE-CIC-IDS2018 capture (DoS-SlowHTTPTest window)
Settings: **Internet ON**, no GPU, no datasets. Use **Save Version -> Save & Run All** (about 1 hour). Every cell is fail-safe.

In [ ]:
import subprocess, os, re, json, time, shutil, traceback
def sh(c, tail=3000):
    print("$", c); r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print(r.stdout[-tail:], r.stderr[-1500:]); return r
sh("pip install -q remotezip cicflowmeter nfstream")
sh("DEBIAN_FRONTEND=noninteractive apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y tshark >/dev/null")
sh("pip show cicflowmeter nfstream | grep -E 'Name|Version'; tshark --version | head -1")
OUT = "/kaggle/working/m7_reextraction"; os.makedirs(OUT, exist_ok=True)
TMP = "/tmp"

In [ ]:
try:
    from remotezip import RemoteZip
    URL = "https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Original%20Network%20Traffic%20and%20Log%20data/Friday-16-02-2018/pcap.zip"
    with RemoteZip(URL) as z:
        PCAP = z.extract("pcap/UCAP172.31.69.25-part1.pcap", path=TMP)
    print("extracted", PCAP, round(os.path.getsize(PCAP) / 1e9, 2), "GB")
    # labelled DoS-SlowHTTPTest window = 10:12-10:58 CSV time = about 14:12-14:58 UTC; keep 14:00-15:10 UTC, TCP port 21
    T0, T1 = 1518789600, 1518793800
    P21 = os.path.join(TMP, "victim_port21_window.pcap")
    sh('tshark -r "{}" -Y "tcp.port==21 && frame.time_epoch >= {} && frame.time_epoch <= {}" -w "{}"'.format(PCAP, T0, T1, P21))
    sh('capinfos -c -a -e "{}"'.format(P21))
except Exception:
    print("!! extraction FAILED"); traceback.print_exc()

In [ ]:
import pandas as pd, urllib.request
try:
    CSV_URL = ("https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/Processed%20Traffic%20Data%20for%20ML%20Algorithms/"
               "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv")
    dst = os.path.join(TMP, "friday.csv")
    if not os.path.exists(dst): urllib.request.urlretrieve(CSV_URL, dst)
    F = pd.read_csv(dst, low_memory=False); F.columns = [c.strip() for c in F.columns]
    F = F[F["Label"] == "DoS attacks-SlowHTTPTest"].copy()
    for c in ["Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "TotLen Fwd Pkts", "TotLen Bwd Pkts", "Dst Port"]:
        F[c] = pd.to_numeric(F[c], errors="coerce")
    REF = {"source": "released CSV (CICFlowMeter, CIC)", "records": len(F),
           "dst_port_21_share": float((F["Dst Port"] == 21).mean()),
           "fwd_pkts_eq1_share": float((F["Tot Fwd Pkts"] == 1).mean()), "bwd_pkts_eq1_share": float((F["Tot Bwd Pkts"] == 1).mean()),
           "zero_payload_share": float(((F["TotLen Fwd Pkts"] + F["TotLen Bwd Pkts"]) == 0).mean()),
           "duration_us_median": float(F["Flow Duration"].median()), "duration_us_p95": float(F["Flow Duration"].quantile(.95)),
           "distinct_(dur,fwd,bwd,bytes)": int(F[["Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "TotLen Fwd Pkts", "TotLen Bwd Pkts"]].drop_duplicates().shape[0])}
    print(REF)
except Exception:
    REF = None; print("!! CSV reference FAILED"); traceback.print_exc()

In [ ]:
def pick(cols, *keys):
    n = {re.sub(r"[^a-z0-9]", "", c.lower()): c for c in cols}
    for k in keys:
        if k in n: return n[k]
    return None
RES = []
try:
    t0 = time.time(); CF = os.path.join(OUT, "cicflowmeter_python_flows.csv")
    r = sh('cicflowmeter -f "{}" -c "{}"'.format(P21, CF))
    C = pd.read_csv(CF, low_memory=False); print("python CICFlowMeter flows:", len(C), "in {:.0f} s".format(time.time() - t0))
    print(list(C.columns)[:40])
    dur = pick(C.columns, "flowduration"); fp = pick(C.columns, "totfwdpkts", "totalfwdpackets")
    bp = pick(C.columns, "totbwdpkts", "totalbackwardpackets", "totalbwdpackets")
    fb = pick(C.columns, "totlenfwdpkts", "totallengthoffwdpackets"); bb = pick(C.columns, "totlenbwdpkts", "totallengthofbwdpackets")
    dp = pick(C.columns, "dstport", "destinationport")
    d = pd.to_numeric(C[dur], errors="coerce")
    RES.append({"source": "re-extracted: cicflowmeter (Python port of CICFlowMeter)", "records": len(C),
                "dst_port_21_share": float((pd.to_numeric(C[dp], errors="coerce") == 21).mean()) if dp else None,
                "fwd_pkts_eq1_share": float((C[fp] == 1).mean()), "bwd_pkts_eq1_share": float((C[bp] == 1).mean()),
                "zero_payload_share": float(((C[fb] + C[bb]) == 0).mean()),
                "duration_us_median": float(d.median()), "duration_us_p95": float(d.quantile(.95)),
                "distinct_(dur,fwd,bwd,bytes)": int(C[[dur, fp, bp, fb, bb]].drop_duplicates().shape[0]),
                "note": "duration in the tool's native unit (check: microseconds in recent versions)"})
    print(RES[-1])
except Exception:
    print("!! python CICFlowMeter FAILED"); traceback.print_exc()

In [ ]:
try:
    from nfstream import NFStreamer
    t0 = time.time()
    N = NFStreamer(source=P21, statistical_analysis=True).to_pandas()
    N.to_csv(os.path.join(OUT, "nfstream_flows.csv.gz"), index=False)
    print("NFStream flows:", len(N), "in {:.0f} s".format(time.time() - t0))
    pay = None
    if "src2dst_bytes" in N and "bidirectional_packets" in N:
        pass
    RES.append({"source": "re-extracted: NFStream (independent flow meter)", "records": len(N),
                "dst_port_21_share": float((N.dst_port == 21).mean()),
                "fwd_pkts_eq1_share": float((N.src2dst_packets == 1).mean()), "bwd_pkts_eq1_share": float((N.dst2src_packets == 1).mean()),
                "zero_payload_share": None,
                "duration_us_median": float(N.bidirectional_duration_ms.median() * 1000), "duration_us_p95": float(N.bidirectional_duration_ms.quantile(.95) * 1000),
                "distinct_(dur,fwd,bwd,bytes)": int(N[["bidirectional_duration_ms", "src2dst_packets", "dst2src_packets", "src2dst_bytes", "dst2src_bytes"]].drop_duplicates().shape[0]),
                "src_ips": int(N.src_ip.nunique()), "top_src_ip": N.src_ip.value_counts().index[0],
                "note": "NFStream counts IP bytes, not payload; duration has millisecond resolution"})
    print(RES[-1])
except Exception:
    print("!! NFStream FAILED"); traceback.print_exc()

In [ ]:
import zipfile
T = pd.DataFrame(([REF] if REF else []) + RES)
T.to_csv(os.path.join(OUT, "m7_reextraction_comparison.csv"), index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); print(T.T.to_string())
with zipfile.ZipFile("/kaggle/working/m7_reextraction_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT):
        if os.path.getsize(os.path.join(OUT, f)) < 90e6: z.write(os.path.join(OUT, f), f)
print("download /kaggle/working/m7_reextraction_outputs.zip")